In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

In [2]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.4f}".format)

In [3]:
DATA_PATH = Path("../data/raw/bank.csv")

df = pd.read_csv(DATA_PATH)

df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


Ячейка 4 — basic checks

In [4]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nDtypes:")
print(df.dtypes)

Shape: (11162, 17)

Columns:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'deposit']

Dtypes:
age          int64
job            str
marital        str
education      str
default        str
balance      int64
housing        str
loan           str
contact        str
day          int64
month          str
duration     int64
campaign     int64
pdays        int64
previous     int64
poutcome       str
deposit        str
dtype: object


Ячейка 5 — create target

In [5]:
target_col = "deposit"

y = df[target_col].map({
    "no": 0,
    "yes": 1,
})

y.head()

0    1
1    1
2    1
3    1
4    1
Name: deposit, dtype: int64

Ячейка 6 — validate target encoding

In [6]:
target_check = pd.DataFrame({
    "original": df[target_col],
    "encoded": y,
})

target_check.head(10)

,original,encoded
0,yes,1
1,yes,1
2,yes,1
3,yes,1
4,yes,1
5,yes,1
6,yes,1
7,yes,1
8,yes,1
9,yes,1


In [7]:
y.value_counts().sort_index()

deposit
0    5873
1    5289
Name: count, dtype: int64

In [8]:
y.value_counts(normalize=True).sort_index()

deposit
0   0.5262
1   0.4738
Name: proportion, dtype: float64

Ячейка 7 — create X and exclude leakage

In [9]:
excluded_cols = [
    "deposit",
    "duration",
]

X = df.drop(columns=excluded_cols)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nExcluded columns:")
print(excluded_cols)

print("\nIs deposit in X?", "deposit" in X.columns)
print("Is duration in X?", "duration" in X.columns)

X shape: (11162, 15)
y shape: (11162,)

Excluded columns:
['deposit', 'duration']

Is deposit in X? False
Is duration in X? False


Ячейка 8 — feature groups

In [10]:
numeric_features = [
    "age",
    "balance",
    "day",
    "campaign",
    "pdays",
    "previous",
]

categorical_features = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "poutcome",
]

expected_features = numeric_features + categorical_features

print("Number of numeric features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))
print("Total expected features:", len(expected_features))
print("Actual X features:", X.shape[1])

print("\nMissing from X:")
print(set(expected_features) - set(X.columns))

print("\nUnexpected in X:")
print(set(X.columns) - set(expected_features))

Number of numeric features: 6
Number of categorical features: 9
Total expected features: 15
Actual X features: 15

Missing from X:
set()

Unexpected in X:
set()


Ячейка 9 — stratified train/test split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (8929, 15)
X_test: (2233, 15)
y_train: (8929,)
y_test: (2233,)


Ячейка 10 — target distribution helper

In [12]:
def target_distribution(series, name):
    counts = series.value_counts().sort_index()
    proportions = series.value_counts(normalize=True).sort_index()
    
    return pd.DataFrame({
        "split": name,
        "class": counts.index,
        "count": counts.values,
        "proportion": proportions.values,
    })

Ячейка 11 — target distribution full/train/test

In [13]:
target_distribution_table = pd.concat(
    [
        target_distribution(y, "full"),
        target_distribution(y_train, "train"),
        target_distribution(y_test, "test"),
    ],
    ignore_index=True,
)

target_distribution_table

,split,class,count,proportion
0,full,0,5873,0.5262
1,full,1,5289,0.4738
2,train,0,4698,0.5262
3,train,1,4231,0.4738
4,test,0,1175,0.5262
5,test,1,1058,0.4738


Ячейка 12 — leakage safety checks

In [14]:
assert "deposit" not in X.columns
assert "duration" not in X.columns

assert "deposit" not in X_train.columns
assert "duration" not in X_train.columns

assert "deposit" not in X_test.columns
assert "duration" not in X_test.columns

assert X_train.shape[0] + X_test.shape[0] == X.shape[0]
assert y_train.shape[0] + y_test.shape[0] == y.shape[0]

print("Leakage checks passed: deposit and duration are excluded from X, X_train, and X_test.")

Leakage checks passed: deposit and duration are excluded from X, X_train, and X_test.


## Stage 3 — Split and baseline classification setup

Task type: binary classification.

Target: `deposit`.

Positive class: `yes`, encoded as `1`.

Negative class: `no`, encoded as `0`.

Main scenario: pre-contact prediction.

Excluded columns:

- `deposit`: target column
- `duration`: leakage-risk feature, known only during/after the call

Validation setup:

- Stratified train/test split
- `test_size=0.2`
- `random_state=42`
- `stratify=y`

Important rule:

The test set is created at this stage but must remain untouched until final evaluation.
All baseline model comparison in this stage will use cross-validation on `X_train` only.

Ячейка — imports for preprocessing and models

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

Ячейка — preprocessing pipelines

In [16]:
numeric_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocessor, numeric_features),
        ("cat", categorical_preprocessor, categorical_features),
    ]
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

Ячейка — models

In [17]:
models = {
    "Dummy most_frequent": DummyClassifier(strategy="most_frequent"),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "LogisticRegression balanced": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    ),
    "DecisionTree max_depth=3": DecisionTreeClassifier(
        max_depth=3,
        random_state=42,
    ),
}

models

{'Dummy most_frequent': DummyClassifier(strategy='most_frequent'),
 'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
 'LogisticRegression balanced': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
 'DecisionTree max_depth=3': DecisionTreeClassifier(max_depth=3, random_state=42)}

Ячейка — CV setup and scoring

In [18]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",  # PR-AUC
}

Ячейка — cross-validation loop

In [19]:
cv_results = []

for model_name, model in models.items():
    clf = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )
    
    scores = cross_validate(
        clf,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1,
    )
    
    row = {"model": model_name}
    
    for metric_name in scoring.keys():
        values = scores[f"test_{metric_name}"]
        row[f"{metric_name}_mean"] = values.mean()
        row[f"{metric_name}_std"] = values.std()
    
    cv_results.append(row)

cv_results_df = (
    pd.DataFrame(cv_results)
    .sort_values("f1_mean", ascending=False)
    .reset_index(drop=True)
)

cv_results_df

,model,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std
0,LogisticRegression balanced,0.7018,0.0096,0.7117,0.0082,0.6228,0.0236,0.6641,0.0150,0.7632,0.0078,0.7634,0.0115
1,LogisticRegression,0.7053,0.0103,0.7431,0.0117,0.5781,0.0225,0.6501,0.0158,0.7631,0.0079,0.7634,0.0115
2,DecisionTree max_depth=3,0.6464,0.0114,0.6363,0.0245,0.6060,0.0903,0.6151,0.0460,0.6969,0.0106,0.6427,0.0085
3,Dummy most_frequent,0.5262,0.0002,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.5000,0.0000,0.4738,0.0002


Ячейка — rounded metrics table

In [20]:
cv_results_rounded = cv_results_df.copy()

metric_cols = [
    col for col in cv_results_rounded.columns
    if col != "model"
]

cv_results_rounded[metric_cols] = cv_results_rounded[metric_cols].round(4)

cv_results_rounded

,model,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std
0,LogisticRegression balanced,0.7018,0.0096,0.7117,0.0082,0.6228,0.0236,0.6641,0.0150,0.7632,0.0078,0.7634,0.0115
1,LogisticRegression,0.7053,0.0103,0.7431,0.0117,0.5781,0.0225,0.6501,0.0158,0.7631,0.0079,0.7634,0.0115
2,DecisionTree max_depth=3,0.6464,0.0114,0.6363,0.0245,0.6060,0.0903,0.6151,0.0460,0.6969,0.0106,0.6427,0.0085
3,Dummy most_frequent,0.5262,0.0002,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.5000,0.0000,0.4738,0.0002


Ячейка — final baseline safety checks

In [21]:
assert "duration" not in X_train.columns
assert "duration" not in X_test.columns
assert "deposit" not in X_train.columns
assert "deposit" not in X_test.columns

print("Stage 3 safety checks passed:")
print("- duration excluded")
print("- deposit excluded")
print("- models evaluated only with CV on X_train")
print("- X_test was not used for model evaluation")

Stage 3 safety checks passed:
- duration excluded
- deposit excluded
- models evaluated only with CV on X_train
- X_test was not used for model evaluation


## Baseline classification conclusions

Models compared with 5-fold StratifiedKFold CV on `X_train` only:

- DummyClassifier(strategy="most_frequent")
- LogisticRegression
- LogisticRegression(class_weight="balanced")
- DecisionTreeClassifier(max_depth=3)

Important constraints:

- `duration` was excluded from all models.
- `X_test` was not used for baseline evaluation.
- No threshold tuning was performed.
- No hyperparameter tuning was performed.
- All preprocessing was placed inside Pipeline / ColumnTransformer.

Initial model selection criterion:

The main baseline comparison focuses on F1, precision, recall, ROC-AUC, and PR-AUC.

Accuracy is reported but not used as the main business metric.

## Stage 3 — Baseline classification conclusions

### Setup

Task type: binary classification.

Target: `deposit`.

Positive class: `yes`, encoded as `1`.

Negative class: `no`, encoded as `0`.

Main scenario: pre-contact prediction.

### Feature matrix

Created:

- `y = deposit`
- `X = df.drop(columns=["deposit", "duration"])`

Excluded columns:

- `deposit`: target column
- `duration`: leakage-risk feature

`duration` is excluded because it is known only during/after the call and is not available at the moment of pre-contact client prioritization.

### Feature groups

Numeric features:

- `age`
- `balance`
- `day`
- `campaign`
- `pdays`
- `previous`

Categorical features:

- `job`
- `marital`
- `education`
- `default`
- `housing`
- `loan`
- `contact`
- `month`
- `poutcome`

### Train/test split

Stratified train/test split was created:

- `test_size=0.2`
- `random_state=42`
- `stratify=y`

Split sizes:

- `X_train`: 8929 rows, 15 features
- `X_test`: 2233 rows, 15 features
- `y_train`: 8929 rows
- `y_test`: 2233 rows

Target distribution was preserved:

- full: class 0 = 52.62%, class 1 = 47.38%
- train: class 0 = 52.62%, class 1 = 47.38%
- test: class 0 = 52.62%, class 1 = 47.38%

Important:

`X_test` was not used for model evaluation in Stage 3.

### Preprocessing

All preprocessing was placed inside `Pipeline` / `ColumnTransformer`.

Numeric preprocessing:

- `SimpleImputer(strategy="median")`
- `StandardScaler()`

Categorical preprocessing:

- `SimpleImputer(strategy="most_frequent")`
- `OneHotEncoder(handle_unknown="ignore")`

`unknown` values were not manually replaced with NaN. They were treated as valid categorical levels.

### Models compared

Models were evaluated with 5-fold `StratifiedKFold` cross-validation on `X_train` only:

- `DummyClassifier(strategy="most_frequent")`
- `LogisticRegression`
- `LogisticRegression(class_weight="balanced")`
- `DecisionTreeClassifier(max_depth=3)`

No hyperparameter tuning was performed.

No threshold tuning was performed.

### CV results

| model | accuracy_mean | precision_mean | recall_mean | f1_mean | roc_auc_mean | average_precision_mean |
|---|---:|---:|---:|---:|---:|---:|
| LogisticRegression balanced | 0.7018 | 0.7117 | 0.6228 | 0.6641 | 0.7632 | 0.7634 |
| LogisticRegression | 0.7053 | 0.7431 | 0.5781 | 0.6501 | 0.7631 | 0.7634 |
| DecisionTree max_depth=3 | 0.6464 | 0.6363 | 0.6060 | 0.6151 | 0.6969 | 0.6427 |
| Dummy most_frequent | 0.5262 | 0.0000 | 0.0000 | 0.0000 | 0.5000 | 0.4738 |

### Baseline conclusion

The strongest simple baseline by F1 is:

`LogisticRegression(class_weight="balanced")`

It achieved:

- F1: 0.6641
- precision: 0.7117
- recall: 0.6228
- ROC-AUC: 0.7632
- PR-AUC: 0.7634

Regular `LogisticRegression` has slightly higher accuracy and precision, but lower recall and F1.

For the current marketing classification framing, the balanced logistic regression is a better first baseline because it improves recall for the positive class while keeping precision reasonable.

### Leakage checks

Passed:

- `duration` excluded from `X`
- `duration` excluded from `X_train`
- `duration` excluded from `X_test`
- `deposit` excluded from `X`
- `deposit` excluded from `X_train`
- `deposit` excluded from `X_test`
- all model evaluation done with CV on `X_train` only
- `X_test` not used for baseline model evaluation
- no preprocessing fitted outside Pipeline/CV